# Aster Payroll Gym — hosted Colab only
Run this notebook in **Google Colab**. Select a GPU for Qwen inference and RL. All start switches are off until you explicitly enable them. No results are prefilled. API phases are skipped by default when keys are unavailable; Colab Qwen inference can still run and judge-dependent scores remain pending. Failures remain visible and do not prevent independent evaluation or evidence export. See `docs/colab.md`.


In [ ]:
import importlib.util
import os
import platform
from pathlib import Path

try:
    found = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    found = False
COLAB_ONLY = platform.system() == "Linux" and Path("/content").is_dir() and found and any(os.environ.get(k) for k in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION"))
if not COLAB_ONLY:
    raise RuntimeError("STOP: this notebook is for hosted Google Colab only. No local fallback.")
print("Recognized hosted Colab. Select a GPU before proceeding.")


## Clone a fixed code revision and install in Colab
Push completed human reviews and the transfer freeze marker first. `REPO_REF` can be a reviewed commit SHA; its resolved `CODE_REVISION` is printed and used for this session. Existing tracked edits are never overwritten. Installation changes only the Colab filesystem.


In [ ]:
assert COLAB_ONLY
import subprocess  # noqa: E402
import sys  # noqa: E402

REPO_URL = "https://github.com/sahajrajmalla/aster-payroll-gym.git"
REPO_REF = "main"  # Prefer the exact reviewed 40-character commit SHA for a final run.
repo = Path("/content/aster-payroll-gym")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
os.chdir(repo)
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("Tracked Colab edits exist. Preserve them before selecting a code revision.")
subprocess.run(["git", "fetch", "origin", REPO_REF], check=True)
CODE_REVISION = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD"], text=True).strip()
subprocess.run(["git", "checkout", "--detach", CODE_REVISION], check=True)
print({"code_revision": CODE_REVISION})
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.19"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--extra", "cloud", "--no-dev"], check=True)


## Storage, credentials and configuration
Mount Drive for checkpoints and results. Add `JUDGE_API_KEY` and `MODEL_API_KEY` in Colab Secrets; both can contain the same Google API key. Confirm model availability, zero-dollar access and disabled billing before setting either free-tier switch. Secrets are never written to the notebook or printed. Keep `SKIP_API_PHASES=True` when keys are unavailable. This skips remote comparisons, judge study and mixed-tier RL without changing the scorer. Qwen inference remains available on a Colab GPU; judge-dependent scores remain pending. Set the switch to `False` only for a separately configured credentialed run.


In [ ]:
assert COLAB_ONLY
import json  # noqa: E402

from google.colab import drive, userdata  # noqa: E402

drive.mount("/content/drive")
def load_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = ""
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)
        print(f"{name}: missing or notebook access disabled")

SKIP_API_PHASES = True  # No-key path: skip API-dependent experiments; retain the original rubric.
for name in ("JUDGE_API_KEY", "MODEL_API_KEY"):
    if SKIP_API_PHASES:
        os.environ.pop(name, None)  # Never reuse a stale secret in deliberate no-key mode.
    else:
        load_secret(name)
if SKIP_API_PHASES:
    print("API phases skipped by choice. Colab Qwen inference is available; RL and judge evidence remain pending.")
os.environ["JUDGE_MODEL"] = "gemini-3.5-flash-lite"  # Verify your account's availability.
FREE_TIER_CONFIRMED = False  # Set True only after checking zero-dollar judge access and disabled billing.
REMOTE_FREE_TIER_CONFIRMED = False  # Independently verify both comparison model configurations.
os.environ["JUDGE_VERIFIED_FREE"] = str(FREE_TIER_CONFIRMED and not SKIP_API_PHASES).lower()
os.environ["JUDGE_MAX_CALLS"] = "100"  # Per-process safety bound; adjust only within verified allowance.
os.environ["JUDGE_MAX_TOKENS"] = "512"  # Bounded allowance for reasoning plus the strict label.
os.environ["JUDGE_REASONING_EFFORT"] = "low"  # Documented support for these Google models.
config = json.loads(Path("configs/train.json").read_text())
config["output_dir"] = "/content/drive/MyDrive/" + ("aster-gym-results-no-api" if SKIP_API_PHASES else "aster-gym-results")
# To reduce compute, use 40 for BOTH sweeps and choose a fresh output_dir.
# config["steps"] = 40
Path("/content/aster-train.json").write_text(json.dumps(config, indent=2))
output_root = Path(config["output_dir"])
output_root.mkdir(parents=True, exist_ok=True)
UV = ["uv", "run", "--frozen", "--extra", "cloud", "--no-dev"]
sys.path.insert(0, str(repo / "src"))
from aster_gym.cloud.workflow import record_skip, run_step  # noqa: E402

STATUS_PATH = output_root / "run_status.json"
def ready_for_judge():
    ready = not SKIP_API_PHASES and FREE_TIER_CONFIRMED and bool(os.environ.get("JUDGE_API_KEY"))
    if not ready:
        print("Pending: verified zero-dollar judge access and JUDGE_API_KEY are required.")
    return ready
print({k: config[k] for k in ("output_dir", "steps", "batch_size", "gradient_accumulation_steps", "beta_values")})


In [ ]:
assert COLAB_ONLY
import shutil  # noqa: E402

if shutil.disk_usage("/content").free < 5 * 1024**3:
    raise RuntimeError("At least 5 GiB of cloud filesystem space is required.")
probe_path = output_root / ".write-test"
probe_path.write_text("cloud-storage-test")
probe_path.unlink()
PREFLIGHT_PATH = output_root / "preflight.json"
GPU_READY = run_step(UV + ["python", "-m", "aster_gym.cloud.preflight", "--output", str(PREFLIGHT_PATH), "--start-preflight"], label="gpu-preflight", status_path=STATUS_PATH, explicit=True)
if PREFLIGHT_PATH.exists():
    print(json.loads(PREFLIGHT_PATH.read_text())["attempts"][-1])
if not GPU_READY:
    print("GPU work is blocked. Remote API evaluation and export remain independent.")
subprocess.run(UV + ["aster-gym", "validate"], check=True)
cohort_code = "from aster_gym.generator import read_taskset,write_taskset; t=read_taskset('data/evaluation.jsonl'); write_taskset(t[:30], 'data/comparison.jsonl'); write_taskset(t[:12], 'data/tool-comparison.jsonl')"
subprocess.run(UV + ["python", "-c", cohort_code], check=True)
transfer_check = "from aster_gym.evidence import verify_transfer_freeze; from aster_gym.generator import read_taskset; verify_transfer_freeze(read_taskset('data/transfer.jsonl'))"
TRANSFER_READY = run_step(UV + ["python", "-c", transfer_check], label="transfer-human-freeze", status_path=STATUS_PATH, explicit=True)


## Three-step smoke
Skipped in no-key mode. Otherwise enable only after GPU and judge checks. Inspect the saved proof and metrics. An unchanged adapter with all equal-reward groups is an ineffective smoke, not a successful update.


In [ ]:
assert COLAB_ONLY
START_SMOKE = False
SMOKE_OK = False
if SKIP_API_PHASES:
    record_skip(label="training-smoke", reason_code="API_PHASES_SKIPPED", status_path=STATUS_PATH, explicit=True)
elif START_SMOKE and GPU_READY and ready_for_judge():
    SMOKE_OK = run_step(UV + ["python", "-m", "aster_gym.cloud.train", "--config", "/content/aster-train.json", "--start-training", "--smoke"], label="training-smoke", status_path=STATUS_PATH, explicit=True)
else:
    print("Training smoke pending: explicitly enable START_SMOKE after preflight.")


In [ ]:
assert COLAB_ONLY
smoke = output_root / "smoke"
for filename in ("config.json", "reference_proof.json", "failure.json"):
    path = smoke / filename
    if path.exists():
        print(filename, json.loads(path.read_text()))
if (smoke / "training_metrics.jsonl").exists():
    print((smoke / "training_metrics.jsonl").read_text())


## Two KL sweeps and held-out comparison
Skipped in no-key mode because the original mixed-tier reward requires a remote judge. This is the long GPU step. Each run starts from the same pinned initial policy. Validation selects beta, then untouched evaluation tasks compare initial and selected policies. Review a successful smoke first. Resume with the exact same code, configuration, splits and output directory. A failed sweep is recorded; unrelated evaluations can continue.


In [ ]:
assert COLAB_ONLY
START_SWEEPS = False
SMOKE_REVIEWED = False  # Set True only after inspecting actual smoke evidence.
RESUME = False
if SKIP_API_PHASES:
    record_skip(label="training-two-beta-sweep", reason_code="API_PHASES_SKIPPED", status_path=STATUS_PATH, explicit=True)
elif START_SWEEPS and SMOKE_REVIEWED and GPU_READY and ready_for_judge():
    command = UV + ["python", "-m", "aster_gym.cloud.train", "--config", "/content/aster-train.json", "--start-training"]
    if RESUME:
        command.append("--resume")
    run_step(command, label="training-two-beta-sweep", status_path=STATUS_PATH, explicit=True)
else:
    print("RL pending: explicitly enable START_SWEEPS after reviewing smoke evidence.")


## Small-model evaluation and tool comparison
These runs are independent of training success and API credentials. In no-key mode, ordinary answers and deterministic rejections are scored normally; correct blocking explanations needing the remote judge remain `pending` with `score: null`. This is partial evidence, not a completed full-rubric comparison. Qwen inference requires Colab CUDA. The frozen cohorts contain 30 shared comparison tasks, all 120 evaluation tasks, 12 tool tasks, and five human-frozen transfer tasks. Every task receives three rollouts. Transfer is skipped until its human freeze verifies.


In [ ]:
assert COLAB_ONLY
START_INFERENCE = False
if START_INFERENCE and GPU_READY:
    if SKIP_API_PHASES:
        print("No remote judge: eligible blocking answers remain pending (JUDGE_UNAVAILABLE); no reward is substituted.")
    cohorts = [("data/comparison.jsonl", "comparison-small", "single"),
               ("data/evaluation.jsonl", "small-model-eval", "single"),
               ("data/tool-comparison.jsonl", "tool-small", "tool")]
    if TRANSFER_READY:
        cohorts.append(("data/transfer.jsonl", "transfer-small", "single"))
    else:
        print("Transfer pending: independent human approval and matching freeze are required.")
    for tasks, name, mode in cohorts:
        run_step(UV + ["python", "-m", "aster_gym.cloud.evaluate", "--config", "/content/aster-train.json", "--tasks", tasks, "--output", str(output_root / name), "--mode", mode, "--start-inference"], label=name, status_path=STATUS_PATH, explicit=True)
else:
    print("Small-model evaluation pending: explicitly enable START_INFERENCE after preflight.")


## Two remote models on the same cohorts
Skipped in no-key mode. For a credentialed run, verify both model IDs in your account. These requests use the shared evaluator and zero-dollar cap. Unknown pricing or absent credentials cannot start calls. Use the same temperature, seed and completion budget as Qwen. Full transcripts, retries, costs and pending judge states are saved on Drive.


In [ ]:
assert COLAB_ONLY
START_REMOTE_EVAL = False
if SKIP_API_PHASES:
    record_skip(label="remote-model-comparison", reason_code="API_PHASES_SKIPPED", status_path=STATUS_PATH, explicit=True)
elif START_REMOTE_EVAL and REMOTE_FREE_TIER_CONFIRMED and ready_for_judge() and os.environ.get("MODEL_API_KEY"):
    eval_config = json.loads(Path("configs/eval.json").read_text())
    eval_config.update(seed=config["seed"], max_tokens=config["max_completion_length"],
                       temperature=config["eval_temperature"], concurrency=1, max_cost_usd=0.0)
    remote_models = [m for m in eval_config["models"] if m["execution"] == "remote_api"]
    if len(remote_models) != 2:
        raise RuntimeError("Exactly two verified remote configurations are required.")
    for model in remote_models:
        model["pricing"] = {"input_per_million": 0, "output_per_million": 0, "verified_free": True}
    eval_path = Path("/content/aster-eval.json")
    eval_path.write_text(json.dumps(eval_config, indent=2))
    for index, model in enumerate(remote_models):
        cohorts = [("data/comparison.jsonl", "comparison", "single"),
                   ("data/tool-comparison.jsonl", "tool", "tool")]
        if TRANSFER_READY:
            cohorts.append(("data/transfer.jsonl", "transfer", "single"))
        for tasks, cohort, mode in cohorts:
            name = f"{cohort}-remote-{index}"
            run_step(UV + ["aster-gym", "eval", "--config", str(eval_path), "--model", model["name"], "--tasks", tasks, "--rollouts", "3", "--mode", mode, "--output", str(output_root / name)], label=name, status_path=STATUS_PATH, explicit=True)
else:
    print("Remote evaluation pending: enable START_REMOTE_EVAL and verify both free configurations and secrets.")


## Judge reliability: fifteen independently labelled outputs, three uncached ratings
Skipped in no-key mode. Complete your human labels before running this cell with credentials. It uses the production eligibility gates and scorer. Ratings are stored in a persistent copy of the review packet. Preserve partial ratings when the provider fails; rerun to resume. Do not replace human labels with model labels.


In [ ]:
assert COLAB_ONLY
START_JUDGE_STUDY = False
judge_packet_path = output_root / "reviews" / "judge-review-packet.json"
if SKIP_API_PHASES:
    record_skip(label="judge-reliability", reason_code="API_PHASES_SKIPPED", status_path=STATUS_PATH, explicit=True)
elif START_JUDGE_STUDY and ready_for_judge():
    source_packet = json.loads(Path("reviews/judge-review-packet.json").read_text())
    rows = source_packet["examples"]
    if len(rows) != 15 or any(type(r.get("human_label")) is not int or r["human_label"] not in (0, 1, 2) or not str(r.get("reviewer") or "").strip() for r in rows):
        print("Judge study pending: fifteen independent human labels and reviewer names are required.")
    else:
        judge_packet_path.parent.mkdir(parents=True, exist_ok=True)
        if not judge_packet_path.exists():
            judge_packet_path.write_text(json.dumps(source_packet, indent=2))
        else:
            stored = json.loads(judge_packet_path.read_text())
            def identity(packet):
                return [(r["example_id"], r["task_input_hash"], r["candidate"], r["human_label"], r["reviewer"]) for r in packet["examples"]]
            if identity(stored) != identity(source_packet):
                raise RuntimeError("Human review changed. Preserve the prior study and select a fresh output directory.")
        run_step(UV + ["aster-gym", "judge-study", "--packet", str(judge_packet_path)], label="judge-reliability", status_path=STATUS_PATH, explicit=True)
else:
    print("Judge study pending: enable START_JUDGE_STUDY after completing independent labels.")


## Export compact evidence
The validated bundle excludes weights, optimizer state, code and pickle. Failed or partial runs remain marked as such. The phase-status file records deliberate API skips. If no model run exists, download only that status file; it is not experiment evidence. The review packet is downloaded separately because it requires your independent labels. Keep checkpoints on Drive; import only JSON evidence on your laptop.


In [ ]:
assert COLAB_ONLY
from google.colab import files  # noqa: E402

archive = "/content/aster-results.zip"
export_code = "from aster_gym.bundles import export_bundle; import sys; print(export_bundle(sys.argv[1], sys.argv[2]))"
has_results = any(output_root.rglob("scores.json"))
if not has_results:
    print("No model experiment exists: only phase status is available, not a results bundle.")
    if STATUS_PATH.exists():
        files.download(str(STATUS_PATH))
elif run_step(UV + ["python", "-c", export_code, str(output_root), archive], label="evidence-export", status_path=STATUS_PATH, explicit=True):
    files.download(archive)
else:
    print("Bundle unavailable: inspect the preserved artifacts; an absent experiment is not a result.")
if PREFLIGHT_PATH.exists():
    files.download(str(PREFLIGHT_PATH))
if judge_packet_path.exists():
    files.download(str(judge_packet_path))
print("Checkpoints remain on Drive. Import the ZIP with aster-gym import-results; copy the reviewed judge packet into reviews/ and regenerate analysis/report.")
